In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Código 4 - VERSIÓN ROBUSTA (TFG PRO)
Ventanas deslizantes consistentes ML/DL sin fugas de información + soporte de lags
"""

import os
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================================
# CONFIGURACIÓN
# ============================================================================

BASE_DIR = os.path.expanduser("/usu/snsaetor/Documents/GitHub/TFGFinal/Datos_TFG_outliers/")

INPUT_ML_TRANSECT = os.path.join(BASE_DIR, "encoded", "ml", "by_transect")
INPUT_DL_TRANSECT = os.path.join(BASE_DIR, "encoded", "dl", "by_transect")

INPUT_ML_GLOBAL = os.path.join(BASE_DIR, "encoded", "ml", "global")
INPUT_DL_GLOBAL = os.path.join(BASE_DIR, "encoded", "dl", "global")

OUTPUT_WINDOWS = os.path.join(BASE_DIR, "windows")

WINDOW_IN = 72
WINDOW_OUT = 72
TARGET_COL = "O3"

# Lags adicionales (opcional)
# Ejemplo: {"NO2": [1,2,3], "TEMP": [1,24]}
LAG_CONFIG = {}  # ← puedes modificar esto

# ============================================================================
# UTILIDADES
# ============================================================================

def add_lags(df, lag_config):
    """Añade lags sin introducir fuga de información."""
    df_lagged = df.copy()

    for col, lags in lag_config.items():
        for lag in lags:
            df_lagged[f"{col}_lag{lag}"] = df_lagged[col].shift(lag)

    return df_lagged


def create_windows(df, window_in, window_out, target_col):
    """
    Genera ventanas de forma CONSISTENTE (una sola vez)
    """
    df = df.sort_index()

    # Eliminar NaNs tras lags
    df = df.dropna()

    values = df.values
    target_idx = df.columns.get_loc(target_col)

    n = len(df)
    n_samples = n - window_in - window_out + 1

    if n_samples <= 0:
        return None, None, None, None

    # Prealocación (eficiente)
    X_dl = np.zeros((n_samples, window_in, values.shape[1]), dtype=np.float32)
    y = np.zeros((n_samples, window_out), dtype=np.float32)
    timestamps = []

    for i in range(n_samples):
        start = i
        end = i + window_in
        out_end = end + window_out

        X_dl[i] = values[start:end]
        y[i] = values[end:out_end, target_idx]
        timestamps.append(df.index[end])

    timestamps = np.array(timestamps, dtype="datetime64[h]")

    # ML = flatten
    X_ml = X_dl.reshape(n_samples, -1)

    return X_ml, X_dl, y, timestamps


def process_file(ml_path, dl_path, output_ml_dir, output_dl_dir, name):
    print(f"\nProcesando {name}...")

    df_ml = pd.read_csv(ml_path, index_col=0, parse_dates=True)
    df_dl = pd.read_csv(dl_path, index_col=0, parse_dates=True)

    # Solo numéricas
    df_ml = df_ml.select_dtypes(include=[np.number])
    df_dl = df_dl.select_dtypes(include=[np.number])

    # Alineación estricta
    df_dl = df_dl.reindex(df_ml.index)

    # Añadir lags SOLO al ML (típico enfoque híbrido)
    df_ml = add_lags(df_ml, LAG_CONFIG)

    # ============================
    # GENERACIÓN ÚNICA DE VENTANAS
    # ============================

    X_ml, X_dl_base, y, timestamps = create_windows(
        df_ml, WINDOW_IN, WINDOW_OUT, TARGET_COL
    )

    if X_ml is None:
        print("  No hay suficientes datos.")
        return

    # ============================
    # RECONSTRUIR X_DL SIN FUGAS
    # ============================

    df_dl = df_dl.loc[df_ml.index]  # asegurar misma longitud tras dropna
    values_dl = df_dl.values

    n_samples = len(X_ml)
    X_dl = np.zeros((n_samples, WINDOW_IN, values_dl.shape[1]), dtype=np.float32)

    for i in range(n_samples):
        X_dl[i] = values_dl[i:i+WINDOW_IN]

    # ============================
    # VALIDACIONES CRÍTICAS
    # ============================

    assert len(X_ml) == len(X_dl) == len(y), "Desalineación detectada"
    assert X_dl.shape[1] == WINDOW_IN, "Error en dimensión temporal"
    assert y.shape[1] == WINDOW_OUT, "Error en horizonte de salida"

    print(f"  Samples: {len(X_ml)}")
    print(f"  X_ml: {X_ml.shape}")
    print(f"  X_dl: {X_dl.shape}")
    print(f"  y: {y.shape}")

    # ============================
    # GUARDADO
    # ============================

    np.save(os.path.join(output_ml_dir, f"{name}_X.npy"), X_ml)
    np.save(os.path.join(output_ml_dir, f"{name}_y.npy"), y)

    np.save(os.path.join(output_dl_dir, f"{name}_X.npy"), X_dl)
    np.save(os.path.join(output_dl_dir, f"{name}_y.npy"), y)

    np.save(os.path.join(output_dl_dir, f"{name}_timestamps.npy"), timestamps)


# ============================================================================
# PROCESAMIENTO
# ============================================================================

def process_folder(input_ml, input_dl, output_ml, output_dl):
    os.makedirs(output_ml, exist_ok=True)
    os.makedirs(output_dl, exist_ok=True)

    ml_files = {f.stem: f for f in Path(input_ml).glob("*.csv")}
    dl_files = {f.stem: f for f in Path(input_dl).glob("*.csv")}

    common = set(ml_files) & set(dl_files)

    for name in sorted(common):
        process_file(
            ml_files[name],
            dl_files[name],
            output_ml,
            output_dl,
            name
        )


# ============================================================================
# MAIN
# ============================================================================

if __name__ == "__main__":
    print("="*60)
    print("VENTANAS ROBUSTAS ML/DL SIN FUGAS")
    print("="*60)

    process_folder(
        INPUT_ML_TRANSECT,
        INPUT_DL_TRANSECT,
        os.path.join(OUTPUT_WINDOWS, "by_transect", "ml"),
        os.path.join(OUTPUT_WINDOWS, "by_transect", "dl")
    )

    process_folder(
        INPUT_ML_GLOBAL,
        INPUT_DL_GLOBAL,
        os.path.join(OUTPUT_WINDOWS, "global", "ml"),
        os.path.join(OUTPUT_WINDOWS, "global", "dl")
    )

    print("\n✔ Proceso completado correctamente.")